# Auto-Quant Experiment Analysis — v0.2.0 multi-strategy event log

Reads `results.tsv` (event log schema: `commit | event | strategy_name | sharpe | max_dd | note`) and produces per-strategy timelines, active-slot utilization, event distributions, and note word frequency.

Run this after the agent has accumulated some rounds of events.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("results.tsv", sep="\t")
df["sharpe"] = pd.to_numeric(df["sharpe"], errors="coerce")
df["max_dd"] = pd.to_numeric(df["max_dd"], errors="coerce")
df["event"] = df["event"].str.strip().str.lower()

# For fork events, strategy_name is "parent→child" — split into two columns
# so we can treat the child as the new strategy moving forward.
def _canonical(name: str) -> str:
    if isinstance(name, str) and "→" in name:
        return name.split("→", 1)[1]
    return name

df["parent"] = df["strategy_name"].map(
    lambda s: s.split("→", 1)[0] if isinstance(s, str) and "→" in s else None
)
df["strategy"] = df["strategy_name"].map(_canonical)

# Give each row a time-ordered index (the row order IS the event order
# because we append as we go). Keep "commit" for reference.
df = df.reset_index(drop=True)
df["round_idx"] = df.index

print(f"Total events: {len(df)}")
print(f"Unique strategies ever seen: {df['strategy'].nunique()}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
# Event type distribution
event_counts = df["event"].value_counts()
print("Events:")
print(event_counts.to_string())

print(f"\nTotal strategies created: {event_counts.get('create', 0) + event_counts.get('fork', 0)}")
print(f"Total strategies killed:  {event_counts.get('kill', 0)}")
print(f"Evolve moves:             {event_counts.get('evolve', 0)}")
print(f"Stable rounds:            {event_counts.get('stable', 0)}")

# Which strategies are still alive at the end?
# A strategy is alive iff its most recent event is NOT 'kill'
last_event_per = df.groupby("strategy").tail(1).set_index("strategy")
alive = last_event_per[last_event_per["event"] != "kill"].index.tolist()
dead = last_event_per[last_event_per["event"] == "kill"].index.tolist()
print(f"\nStill alive at end ({len(alive)}): {alive}")
print(f"Killed during run  ({len(dead)}): {dead}")

In [ ]:
# Peak sharpe per strategy (across its lifetime)
peak = df.dropna(subset=["sharpe"]).groupby("strategy")["sharpe"].max().sort_values(ascending=False)
print("Peak Sharpe per strategy:\n")
for name, s in peak.items():
    last_evt = last_event_per.loc[name, "event"]
    status = "☠ killed" if last_evt == "kill" else "✓ alive"
    print(f"  {status}  {name:30s}  peak_sharpe={s:.4f}")

## Per-strategy Sharpe trajectories

Each active line = one strategy's sharpe over its lifetime. Gaps indicate
`stable` rounds where the strategy wasn't modified. A line ending in a dot
means the strategy was killed at that point. Scatter markers by event type
let you see where the agent chose to evolve / fork / kill each strategy.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

event_marker = {"create": "o", "evolve": ".", "fork": "^", "stable": ",", "kill": "x"}

for name, g in df.dropna(subset=["sharpe"]).groupby("strategy"):
    g = g.sort_values("round_idx")
    ax.plot(g["round_idx"], g["sharpe"], alpha=0.5, label=name)
    for evt, marker in event_marker.items():
        sub = g[g["event"] == evt]
        if len(sub) and evt != "stable":  # don't clutter with stable markers
            ax.scatter(sub["round_idx"], sub["sharpe"], marker=marker, s=50, alpha=0.9)

# Mark kills explicitly (last-event-of-strategy with no sharpe)
for name, g in df.groupby("strategy"):
    last = g.iloc[-1]
    if last["event"] == "kill":
        # put an x at y=0 at the kill position to signal strategy ended
        ax.scatter([last["round_idx"]], [0], marker="x", c="red", s=80)

ax.axhline(0, color="gray", linewidth=0.5, alpha=0.5)
ax.set_xlabel("event # (chronological)")
ax.set_ylabel("sharpe")
ax.set_title("Per-strategy Sharpe trajectories")
ax.legend(loc="best", fontsize=8)
ax.grid(alpha=0.3)
plt.show()

## Active strategy count over time

With a hard cap of 3, the count should mostly sit at 3 (agent keeps slots full)
and occasionally dip to 2 when a strategy is killed without immediate replacement.
Sustained dips below 3 mean the agent isn't filling the cap — either exploring
cautiously or running out of ideas.

In [ ]:
# Walk the event log, maintain a set of active strategies, record count per event.
# `fork` is counted as a create for the child (+1), `kill` as -1, create as +1.
active = set()
counts = []
for _, row in df.iterrows():
    name = row["strategy"]
    evt = row["event"]
    if evt == "create" or evt == "fork":
        active.add(name)
    elif evt == "kill":
        active.discard(name)
    # evolve / stable don't change membership
    counts.append(len(active))

df["active_count"] = counts

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(df["round_idx"], df["active_count"], drawstyle="steps-post")
ax.axhline(3, color="red", linestyle="--", alpha=0.3, label="cap (3)")
ax.set_xlabel("event #")
ax.set_ylabel("active strategies")
ax.set_title("Cap utilization over time")
ax.set_ylim(0, 4)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Note word frequency

Rough proxy for what paradigms, indicators, and failure modes the agent
thought about during this run. Skim the top 30 — if you see only one
paradigm family dominating, anchoring is creeping in.

In [ ]:
from collections import Counter
import re

text = " ".join(df["note"].dropna().astype(str).str.lower().tolist())
words = re.findall(r"[a-z]{3,}", text)
stop = {
    "the", "and", "for", "with", "this", "that", "from", "was", "too",
    "add", "added", "use", "using", "but", "all", "not", "has", "have",
    "trade", "trades", "run", "ran", "same", "than", "more", "less",
    "still", "then", "one", "two", "new", "old", "now",
}
top = Counter(w for w in words if w not in stop).most_common(30)
print("Top 30 words across all notes:")
for w, c in top:
    print(f"  {c:4d}  {w}")